# 00 — Build the data bundle (plan.md Stage 2, task T1)

Downloads the public datasets once and freezes them as question lists, images and per-token background masks.

**Kaggle settings:** Accelerator **None** (CPU — uses no GPU quota) · Internet **on** · secret `cvpr` = Hugging Face token.

**Output:** `/kaggle/working/medprune-data` (~0.7 GB). After the run: *Save Version* → Output → *New Dataset*
`medprune-data` (private) → *Settings → Sharing*: add your other two Kaggle accounts. RadImageNet-VQA is **not**
included (license); every GPU session builds it itself.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("cvpr")

Code goes to `/tmp` so it does not end up inside the dataset; only `medprune-data` is written to `/kaggle/working`.

In [ ]:
!rm -rf /tmp/cvpr && git clone --depth 1 https://github.com/pushkar1887/cvpr /tmp/cvpr
%cd /tmp/cvpr
!git log --oneline -1
!bash scripts/setup_env.sh

In [ ]:
import shutil

!df -h /tmp /kaggle/working
free_gb = shutil.disk_usage("/tmp").free / 2**30
assert free_gb >= 25, f"only {free_gb:.0f} GB free in /tmp; the OmniMedVQA zip alone is 10.7 GB"

Questions and images (≈ 15–25 min; the OmniMedVQA zip is downloaded to `/tmp`). Prints the counts and OmniMedVQA's modality names.

In [ ]:
!python -m medprune.data build --out /kaggle/working/medprune-data --work /tmp/medprune-work --skip radimagenet

Background masks for all three models, plus the overlays used for eyeballing (≈ 5–10 min).

In [ ]:
!python -m medprune.data masks --bundle /kaggle/working/medprune-data

In [ ]:
!python -m medprune.data summary --bundle /kaggle/working/medprune-data > /tmp/summary.json && echo written

### Checks against plan.md §C2

In [ ]:
import json
import pandas as pd

summary = json.load(open("/kaggle/working/medprune-data/bundle_manifest.json"))
expected = {"slake_en": 1061, "slake_zh": 1033, "omni_panel": 1200, "natural": 1000, "omni_anat": 3329}
rows = []
for name, info in summary["datasets"].items():
    rows.append({"dataset": name, "questions": info["questions"], "images": info["images"],
                 "expected": expected.get(name), "match": info["questions"] == expected.get(name)})
display(pd.DataFrame(rows))
print("slake_en images by modality:", summary["datasets"]["slake_en"]["by_modality"])

In [ ]:
bg = []
for name, info in summary["datasets"].items():
    for model, by_modality in info["median_background_fraction"].items():
        for modality, value in by_modality.items():
            bg.append({"dataset": name, "model": model, "modality": modality, "median_bg": value})
display(pd.DataFrame(bg).pivot_table(index=["dataset", "modality"], columns="model", values="median_bg"))

### Overlays (red = background tokens). Check: table removed, lungs kept, dermoscopy fully foreground.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

overlay_root = Path("/kaggle/working/medprune-data/masks/overlays")
modalities = sorted(p for p in overlay_root.iterdir() if p.is_dir())
fig, axes = plt.subplots(len(modalities), 3, figsize=(9, 3 * len(modalities)), squeeze=False)
for row, folder in zip(axes, modalities):
    for ax, path in zip(row, sorted(folder.glob("*.png"))[:3]):
        ax.imshow(Image.open(path))
    for ax in row:
        ax.set_axis_off()
    row[0].set_title(folder.name, loc="left")
plt.tight_layout()

In [ ]:
!rm -rf /tmp/medprune-work && du -sh /kaggle/working/medprune-data

### Next
1. *Save Version* (Save & Run All) so the output is kept; create the private dataset `medprune-data` from the output;
   share it with your other two Kaggle accounts.
2. Copy `bundle_manifest.json` into the repo as `configs/bundle_manifest.json`, commit, and add a Work log entry with
   the counts above (plan.md Part G).